# Climate Indices from Daily Products

This tutorial explains the annual index functions with tiny synthetic arrays so it stays offline and fast.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

import sys
import numpy as np
import pandas as pd
import xarray as xr

CLIMATE_REPO = REPOS / "mia-climate"
sys.path.insert(0, str(CLIMATE_REPO))

from lib.indices_temperature import TEMPERATURE_INDEX_SPECS, compute_index as compute_temperature_index
from lib.indices_precipitation import PRECIPITATION_INDEX_SPECS, compute_precipitation_index


In [ ]:
pd.DataFrame([s.__dict__ for s in TEMPERATURE_INDEX_SPECS + PRECIPITATION_INDEX_SPECS])


In [ ]:
time = np.array([
    "2000-01-01", "2000-07-01", "2000-12-31",
    "2001-01-01", "2001-07-01", "2001-12-31",
], dtype="datetime64[ns]")

def da(values, name, units):
    return xr.DataArray(
        np.array(values, dtype="float32"),
        dims=("time",),
        coords={"time": time},
        name=name,
        attrs={"units": units},
    )

daily_temperature = {
    "tmax": da([25, 38, 22, 27, 35, 24], "tmax", "degC"),
    "tmin": da([15, 25, 12, 17, 23, 14], "tmin", "degC"),
    "tmean": da([20, 32, 18, 22, 28, 19], "tmean", "degC"),
}

daily_precipitation = {
    "pr": da([0, 12, 2, 5, 30, 0], "pr", "mm"),
}


In [ ]:
temperature_outputs = {
    spec.index_id: compute_temperature_index(spec.index_id, daily_temperature)
    for spec in TEMPERATURE_INDEX_SPECS
}

pd.DataFrame({k: v.values for k, v in temperature_outputs.items()}, index=[2000, 2001])


In [ ]:
precipitation_outputs = {
    spec.index_id: compute_precipitation_index(spec.index_id, daily_precipitation)
    for spec in PRECIPITATION_INDEX_SPECS
}

pd.DataFrame({k: v.values for k, v in precipitation_outputs.items()}, index=[2000, 2001])
